# 📊 Gerador de Painel de Cobrança via WhatsApp Web

Este notebook processa uma planilha de inadimplentes (`.xls` ou `.xlsx`) e gera uma página **HTML autônoma** com:
* Links diretos para o **WhatsApp Web** com mensagem personalizada (singular/plural) e cálculo automático de valores;
* **Controle de contatos** (flag `Contatado?`) com cards dinâmicos de acompanhamento em quantidade, percentual e valor em R$;
* **Campo de observação** persistente e opção para salvar uma cópia do HTML com as anotações embutidas;
* **Exportação limpa para Excel (`.xlsx`)** sem a coluna de links.

---

### 🚀 Preencha os dados e execute a célula abaixo
1. Informe o **Nome do Atendente** e o **Nome da Empresa** nos campos do formulário (ou deixe em branco para os padrões).
2. Clique no botão **Play (▶)** ao lado da célula.
3. Selecione a planilha `.xls` ou `.xlsx` quando o botão **Escolher arquivos** aparecer.

In [ ]:
#@title Configuração e Execução
NOME_ATENDENTE = "" #@param {type:"string"}
NOME_EMPRESA = "" #@param {type:"string"}

import urllib.request
from pathlib import Path

atendente_final = NOME_ATENDENTE.strip() or "Nome do atendente"
empresa_final = NOME_EMPRESA.strip() or "Nome da empresa"

print(f"Configurado -> Atendente: '{atendente_final}' | Empresa: '{empresa_final}'\n")

SCRIPT_LOCAL = "gerar_inadimplentes_whatsapp_colab.py"
URL_RAW_SCRIPT = "https://raw.githubusercontent.com/amisadaimodesto/gerador-cobranca-whatsapp/main/gerar_inadimplentes_whatsapp_colab.py"

if not Path(SCRIPT_LOCAL).exists():
    print("Baixando motor de processamento do GitHub...")
    urllib.request.urlretrieve(URL_RAW_SCRIPT, SCRIPT_LOCAL)

import gerar_inadimplentes_whatsapp_colab as gerador

if gerador.esta_no_google_colab():
    caminho_planilha = gerador.selecionar_arquivo_colab()
else:
    caminho_planilha = input("Informe o caminho completo da planilha (.xls ou .xlsx): ").strip().strip('"').strip("'")

linhas, nome_aba = gerador.ler_planilha(caminho_planilha)
registros = gerador.extrair_registros(linhas)

if not registros:
    raise RuntimeError("Não foi possível localizar os blocos de Inadimplentes na planilha.")

conteudo_html = gerador.gerar_html(
    registros,
    nome_aba,
    nome_atendente=atendente_final,
    nome_empresa=empresa_final,
)

arquivo_saida = gerador.salvar_html(conteudo_html, gerador.NOME_ARQUIVO_SAIDA)

print(f"\n✅ Concluído! {len(registros)} registro(s) processado(s).")
print(f"📄 Arquivo gerado em: {arquivo_saida}")

gerador.tentar_baixar_no_colab(arquivo_saida)